# March Mania · Joint win-loss strength
**Milestone 10 — preserve evidence → build a different strength representation → test fixed comparisons.**

Round09's bracket context worsened mean Brier by **0.0004021** beyond its seed-status control, with improvement in two of four seasons. No expansion or feature promotion follows that result.

This notebook focuses on men. It builds **two candidates** from regular-season wins, losses, opponents and venue: joint Bradley–Terry ability and a model-conditional uncertainty correction. Neither is a new statistical method. The classifier remains the same compact logistic reference.

At most **seven new season-rating fits and eight new tournament-classifier fits**. Four references and seven base snapshots are reused. All 2016–2019 validation seasons are repeatedly used exploratory history—not untouched tests, and not the 2026 leaderboard. Read RESEARCH_PLAN.md before interpreting results.

**Verified installation:** Before any project fitting, a fresh subprocess checks the installed solver path and source, its imported function code, and bounded refinement on synthetic analytic arrays. The update ID must be `bt-runtime-verified-20260911`. Stop if this check fails. The prior accepted rating checkpoints are retained. No acceptance tolerance or feature recipe changed.


In [ ]:
from pathlib import Path
import os, sys, json
import pandas as pd
import plotly.io as pio
from IPython.display import display, FileLink
KIT = Path.cwd().resolve()
if not (KIT / 'run_round10.py').is_file():
    KIT = Path.home() / 'march_win_strength'
assert (KIT / 'run_round10.py').is_file(), 'Open this notebook inside march_win_strength.'
sys.path.insert(0, str(KIT))
import subprocess
UPDATE_ID = 'bt-runtime-verified-20260911'
def verify_active_solver():
    check = subprocess.run(
        [sys.executable, str(KIT / 'apply_win_strength_update.py'),
         '--kit', str(KIT), '--expected-update-id', UPDATE_ID, '--verify-only'],
        cwd=KIT, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        timeout=60)
    print(check.stdout, end='')
    if check.returncode:
        raise RuntimeError('Installed solver check failed. Do not train. Preserve reports/runtime_installation_return.zip.')
verify_active_solver()
from run_round10 import run_stage as _bounded_run_stage
def run_stage(stage, max_seconds):
    verify_active_solver()
    return _bounded_run_stage(stage, max_seconds=max_seconds)
from strength_plots import figures
pio.renderers.default = 'plotly_mimetype'
print('Kernel:', sys.executable)
print('Research kit:', KIT)
print('No source repository, AWS service, GitHub, or environment changes are made.')

## 1. Keep the previous negative result
Positive Brier change is worse. The prior gate is not loosened and the bracket feature is not used in this experiment.

In [ ]:
prior = pd.read_csv(KIT / 'evidence/round09/ablations.csv')
display(prior.query("comparison=='scaled_given_status'").round(7))
print(json.dumps(json.loads((KIT / 'evidence/round09/decisions.json').read_text()), indent=2))

## 2. Fit season-local ratings, then freeze matchup features
For regular-season game $i$ versus $j$:
$$P(i\text{ wins})=\operatorname{sigmoid}(a_i-a_j+h\,\mathrm{venue}_{ij}).$$
The fit minimizes the sum of physical-game negative log likelihoods plus fixed Gaussian-prior penalties. Team prior SD is 2, home-effect prior SD is 1. Score margins, tournament outcomes and ranks do not enter this rating likelihood. Venue affects the regular-season fit; new matchup predictions set venue to neutral.

**Candidate 1:** $d=a_i-a_j$.

**Candidate 2:** $\operatorname{logit}(E[\operatorname{sigmoid}(Z)])-d$, where $Z\sim N(d,v)$ and $v=V_{ii}+V_{jj}-2V_{ij}$. $V$ comes from the inverse penalized Hessian, including the home-effect nuisance parameter. This is approximate, model-conditional uncertainty—not proven empirical coverage. Numerical integration checks 64 against 128 Gauss–Hermite nodes.

Seven day-132 snapshots are fitted and saved separately. Potential matchups are constructed before tournament outcomes are attached. A snapshot's same-season regular-season data are legal for that season's pre-tournament forecast. This is not a pregame regular-season backtest. **Preparation ceiling: 300 seconds; heartbeat every 15 seconds.**

In [ ]:
run_stage('prepare', max_seconds=300)
RUN = Path(json.loads((KIT / 'reports/latest_run.json').read_text())['run_dir'])
print(json.dumps(json.loads((RUN / 'prepare.json').read_text()), indent=2))
display(pd.read_csv(RUN / 'prior_replay.csv').round(7))
rating_diagnostics = pd.read_csv(RUN / 'rating_diagnostics.csv')
display(rating_diagnostics.reindex(columns=['Season','teams','regular_games','graph_components','iterations','initial_gradient_max_abs','newton_steps','gradient_max_abs','covariance_inverse_error','solver_path']))
print('Missing refinement fields on a migrated rating mean that the original fit passed without this new continuation step.')

In [ ]:
display(pd.read_csv(RUN / 'feature_registry.csv').query('new_candidate == True'))
display(pd.read_csv(RUN / 'coverage.csv'))
profiles = pd.read_csv(RUN / 'team_profiles.csv')
display(profiles.query('Season == 2019').sort_values('bt_ability', ascending=False).head(15)[
    ['TeamID','bt_ability','bt_marginal_sd','regular_games','regular_wins','unique_opponents','strength','seed']])
print('Team profiles are descriptive inputs, not evidence of tournament performance.')

## 3. Test the contribution without changing the classifier
| Configuration | Inputs | Interpretation |
|---|---:|---|
| anchor | 16 | Four verified saved references |
| anchor_bt | 17 | Primary ability test |
| anchor_bt_uncertainty | 18 | Secondary conditional-uncertainty test |

Train on 2013 through the year before each validation season. Validate on 2016, 2017, 2018 and 2019 main-draw games. The original logistic C=0.1, no intercept, mirrored orientations, physical-game weighting and train-only scaling stay fixed. Only training-constant inputs may be removed. The uncertainty-alone ablation is deferred until conditional usefulness is demonstrated.

**Eight new classifiers, no classifier search. Evaluation ceiling: 180 seconds.**

In [ ]:
run_stage('evaluate', max_seconds=180)
metrics = pd.read_csv(RUN / 'metrics.csv')
display(metrics[['Gender','Season','recipe','games','train_games','brier','log_loss','delta_vs_anchor','source']].round(7))
display(pd.read_csv(RUN / 'ablations.csv').round(7))

## 4. Apply the registered decision rule
**Primary:** `ability_given_anchor`. **Secondary:** `uncertainty_given_ability`.

For either comparison, consider an unchanged later-era test only if mean Brier change ≤ −0.0005, at least 3/4 seasons improve, and worst deterioration ≤ +0.003. These are resource-allocation thresholds, not statistical significance. The primary is not replaced by a favorable secondary result. Nothing runs automatically after this experiment.

A technical COMPLETE does not mean the features worked. Check log loss, coefficient stability and redundancy, not only the smallest Brier.

In [ ]:
print(json.dumps(json.loads((RUN / 'decisions.json').read_text()), indent=2))
display(pd.read_csv(RUN / 'training_overlap.csv').round(5))
print(json.dumps(json.loads((RUN / 'evaluation_receipt.json').read_text()), indent=2))

## 5. Plotly evidence
Ten charts: prior result, win-rate and margin-strength comparisons, model uncertainty and schedule support, home-effect estimates, per-season Brier, controlled deltas, uncertainty correction, calibration, and training-only overlap. Use hover for exact values. The charts do not establish causality or calibrated posterior coverage.

In [ ]:
plots = figures(RUN, KIT / 'evidence/round09')
assert len(plots) == 10
for fig in plots[:5]:
    fig.show()

In [ ]:
for fig in plots[5:]:
    fig.show()

## 6. Export and stop this milestone
**Report ceiling: 120 seconds.** The archive is exported only after source, raw-input and upstream-artifact preservation checks pass. It excludes rating models/covariance, classifiers, raw rows and per-game predictions. Save this executed notebook with Ctrl+S; return `reports/milestone_10_return.zip` to ChatGPT.

Feature engineering remains open. A gain here needs an unchanged later-era and stronger-production-recipe check. A failure is retained as negative evidence and is not automatically retried with different priors.

In [ ]:
run_stage('report', max_seconds=120)
record = json.loads((KIT / 'reports/latest_report.json').read_text())
print('Return ZIP:', record['return_zip'])
print('Interactive HTML:', record['html'])
display(FileLink(str(Path(record['return_zip']).relative_to(KIT))))
display(FileLink(str(Path(record['html']).relative_to(KIT))))
print('Save this notebook. Keep all private_runs folders. Stop this milestone here.')